# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [ ]:
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


def clean_text(text):

    #noktalamaları kaldırdık
    for punctuation in string.punctuation:
        text = text.replace(punctuation, ' ')
    #küçük harflerle yazdık
    lowercase_text = text.lower()

    #kelimelere ayırdık
    tokens = word_tokenize(lowercase_text)

    #sayıları kaldırdık
    alphabetic_words = [word for word in tokens if word.isalpha()]

    # anlamı olmayan kelimeleri kaldırdık
    stop_words = set(stopwords.words('english'))

    filtered_words = [
        word for word in alphabetic_words
        if word not in stop_words
    ]

    # kelimeleri anlamlı en basit haline getirdik
    lemmatizer = WordNetLemmatizer()

    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in filtered_words
    ]

    # kelimeleri birleştirdik
    cleaned_text = ' '.join(lemmatized_words)

    return cleaned_text

data['clean_text'] = data['text'].apply(clean_text)
data.head()


,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gld cunixb cc columbia edu gary l dare subject...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep vela ac oakland edu cardinal ximenez ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,miner kuhub cc ukans edu subject ancient book ...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlep vela ac oakland edu cardinal ximenez ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivov superior carleton ca vladimir zhivov s...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


vectorizer = CountVectorizer()

document_matrix = vectorizer.fit_transform(data['clean_text'])

# LDA model
lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

lda_model.fit(document_matrix)
topic_distribution = lda_model.transform(document_matrix)

print("Document matrix shape:", document_matrix.shape)
print("Topic distribution shape:", topic_distribution.shape)

Document matrix shape: (1199, 17271)
Topic distribution shape: (1199, 5)


##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [14]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [15]:
print_topics(lda_model, vectorizer)

Topic 0:
[('edu', 800.419085612616), ('game', 679.1422290904557), ('team', 629.7655379099492), ('line', 467.7517743976809), ('subject', 410.3707671582985), ('hockey', 407.3548704949053), ('organization', 397.441778397986), ('year', 397.119942172473), ('go', 368.50510613966406), ('season', 297.9339913816419)]
Topic 1:
[('edu', 407.18027735949283), ('subject', 255.61435983953567), ('god', 253.66447452298655), ('line', 243.66441143439192), ('organization', 224.68244670190904), ('would', 173.21275934777282), ('one', 152.0375825582965), ('com', 149.71306728225878), ('university', 143.19682326480762), ('church', 121.64452400859723)]
Topic 2:
[('pt', 315.6830254207129), ('la', 231.70824447004048), ('period', 213.70737745881976), ('edu', 205.54466285473393), ('new', 175.97424312967087), ('play', 146.48165211188294), ('team', 142.1317431128341), ('chi', 137.1984249309607), ('game', 133.5694842687447), ('bos', 133.1718773562438)]
Topic 3:
[('ca', 369.9502268673211), ('would', 290.05343517002126)

## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [17]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [18]:
cleaned_example = [clean_text(text) for text in example]

example_vector = vectorizer.transform(cleaned_example)

predicted_topics = lda_model.transform(example_vector)

for topic_index, probability in enumerate(predicted_topics[0]):
    print(f"Topic {topic_index}: {probability:.4f}")

dominant_topic = predicted_topics[0].argmax()

print(f"\nDominant Topic: {dominant_topic}")

Topic 0: 0.9326
Topic 1: 0.0168
Topic 2: 0.0168
Topic 3: 0.0170
Topic 4: 0.0168

Dominant Topic: 0


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!